# Level 0 multi-seed diagnostics

Mean and standard-deviation bands are descriptive; use paired confirmation seeds for optimizer claims.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getenv('NANOGPT_LEVEL0_RESULTS_ROOT', '/tmp/nanogpt-level0-bpe/results'))
rows = []
selected_rows = []
for path in sorted(ROOT.glob('*_seed_*/metrics.csv')):
    optimizer, seed_text = path.parent.name.rsplit('_seed_', 1)
    frame = pd.read_csv(path)
    frame['optimizer'] = optimizer
    frame['seed'] = int(seed_text)
    rows.append(frame)
    selected_path = path.parent / 'selected_checkpoint_metrics.json'
    if selected_path.exists():
        selected = json.loads(selected_path.read_text())
        selected.update({'optimizer': optimizer, 'seed': int(seed_text)})
        selected_rows.append(selected)
all_df = pd.concat(rows, ignore_index=True)
selected_df = pd.DataFrame(selected_rows)
all_df.groupby('optimizer').seed.nunique(), selected_df


In [ ]:
def band_plot(metric, ylabel=None):
    fig, ax = plt.subplots(figsize=(10, 5))
    for optimizer, data in all_df.groupby('optimizer'):
        aggregate = data.groupby('step')[metric].agg(['mean','std']).reset_index()
        std = aggregate['std'].fillna(0)
        line, = ax.plot(aggregate.step, aggregate['mean'], label=optimizer)
        ax.fill_between(aggregate.step, aggregate['mean']-std, aggregate['mean']+std, alpha=.2, color=line.get_color())
    ax.set(xlabel='optimizer step', ylabel=ylabel or metric, title=f'{metric}: mean +/- 1 standard deviation')
    ax.grid(alpha=.25); ax.legend(); plt.show()

for metric in ['val_loss','val_perplexity','val_accuracy','val_generalization_gap']:
    band_plot(metric, 'accuracy fraction' if metric == 'val_accuracy' else metric)


In [ ]:
ww_rows = []
for run in sorted(ROOT.glob('*_seed_*')):
    optimizer, seed_text = run.name.rsplit('_seed_', 1)
    for path in run.glob('weightwatcher_step_*.csv'):
        frame = pd.read_csv(path)
        frame['optimizer'] = optimizer
        frame['seed'] = int(seed_text)
        ww_rows.append(frame)
if ww_rows:
    ww = pd.concat(ww_rows, ignore_index=True)
    ww['alpha'] = pd.to_numeric(ww['alpha'], errors='coerce')
    ww = ww[np.isfinite(ww.alpha)]
    for matrix_type, matrix_data in ww.groupby('matrix_type'):
        fig, ax = plt.subplots(figsize=(10, 4))
        for optimizer, optimizer_data in matrix_data.groupby('optimizer'):
            per_seed = optimizer_data.groupby(['seed','step']).alpha.mean().reset_index()
            aggregate = per_seed.groupby('step').alpha.agg(['mean','std']).reset_index()
            std = aggregate['std'].fillna(0)
            line, = ax.plot(aggregate.step, aggregate['mean'], label=optimizer)
            ax.fill_between(aggregate.step, aggregate['mean']-std, aggregate['mean']+std, alpha=.2, color=line.get_color())
        ax.axhline(2.0, linestyle='--', linewidth=1)
        ax.set(xlabel='optimizer step', ylabel='WeightWatcher alpha', title=matrix_type)
        ax.grid(alpha=.25); ax.legend(); plt.show()
